# NOTICE

I have generalized the hierarchy of mf6lab. The three levels (explained below) now each has a `src` folder to hold the .py files at these levels. There is now also a `notebooks` folder with notebooks that essentially show and explain in steps how the `.py` files are supposed to work.

The local parameter and package selection workbook should be placed in the local `data` directory.

This may not yet be correctly done for the older projects, but will be corrected in the future.

The GGOR project is as it should be.

#TO 251119

# `mf6lab` succinctly explained

`mf6lab` facilitates simulating groundwater with `Modflow 6` using `python` and `flopy`. It was developed under the name `mflab` between 2008 and 2015 in Matlab. Development of `mflab` seized in 2016; I then switched developing `mf6lab` in python (with `flopy` and `Modflow 6`) in 2016. The main reason is that `python` is free to use of my students and anyone else around the world. `flopy` is a python-written package to generate input for Modlfow 6, run it and analyze its output. It's developed by a fairly large highly committed team and has become so advanced that it does no longer make sense to develop the functionality myself as I did in the old `mflab`.

The old `mflab` and current `mf6lab` projects can be found on `github` under `Theo Olsthoorn`

To facilitate creating projects and cases within such projects mf6lab has the following directory strurture:

The hierarchy shows three distict levels: mf6lab, individual projects in the folder `mf6lab/Projects` and individual cases under `mf6lab/Projects/<specific project>/cases`

Each of these three levels has subfolder `src` to store code (python `.py` files) pertaining to the level within the `mf6lab` hierarchy (e.g. `mf6tools.py` in `mf6lab/src`, `GGOR_tools.py` and `mf_setup.py` in `mf6lab/Projects/GGOR/src` and `mf_adapt.py`, `mf_run.py` and `mf_analyze.py `in `mf6lab/Projects/GGOR/cases/AAN_ZGK/src`). The projects and the cases have more subfolders such as `doc`, `data`, `images`, `notebooks`, while the cases also have the folders `SIM`, `GWF`, `GWT` and `MP7` to store the input and output files pertaining to each mf6 simulation, mf6's Ground Water Flow model and mf6's Groundwater Transport Model. Further directories and subdirectories may be added as needed, e.g. `data/meteo` and `data/GIS`, `videos`.

# How to run a groundwater model in mf6lab?

From the actual <case> directory run `mf_run.py`. This will run mf6lab's backbone `mf_setup.py` setting up the simulation. `mf_setup.py` reads which models and packages to use from an Excel workbook `<case_name>.xlsx`. It also reads default parameters for the `GWF` and `GWT` models from it and puts these in a large dictionary of which the keys are the `GWF` and `GWT` packages. mf_setup then imports `mf_adapt.py` to adapt the paramters to meet the demands of the current case. `mf_adapt.py` imports in turn local `settings.py` to get user parameters for the run itself. Then flopy will create the input files, after which Modlfow 6 is called to run. Modflow will read its fles form the local `SIM`, `GWF` and possibly `GWT` directories, carries out the simulation, and finishes with a message indicating success or failure.

If failure, look in the list files in the `SIM`, `GWF` and `GWT` directories to find what went wrong. Adapt the input and try again.

If success, then local `mf_analyze.py` is run by the user. The purpose of mf_analyze.py is to read the binary output produced by `mf6` analyze it and present is in tables and figures.


## The Excel Workbook `<case_name>.xlsx`

Each case has its own Excel workbook named <case>.xlsx which tells what packages mf6 should use and which models (`GWF` and or `GWT`) to run and further specifies default parmeter values for all `SIM`, `GWF`, `GWT` and `MF7` (`Modpath 7`) packages. Next to this, the workbook allows defining stress periods and layer properties that are constant within a layer.

The Excel workbook has a number of spreasheets with the following names:

`NAM`, `SIM6`, `GWF6`, `GWT6`, `PER`, `LAY`

### The sheet `NAM`

Shows all possible packages of the groundwater flow and the groundwater transport models of Modflow 6 in the form of `Gwf???` and `Gwt???` in which Gwf stands for `Ground Water FLow` and `Gwt` stands for `Ground Water Transport` and where `???` is the mostly 3-character acronym of the package. So we get package names like `Gwfdis`, `Gwfwel`, `Gwfchd` etc as well as `Gwtdis` `Gwtadv` etc. One designates a package to be using in a given simulation by selecting it. Is is done by placing a 1 instead of a 0 in column `ON/OFF`. Selected packages are indicted in the column `Indicator` (column C). Selection of at least one package starting with `Gwf` implies that the Modflow flow model will be run and selecting at least one packages starting with `Gwt` ensures that the groundwater transport model will be run. Any combination will cause both models to be run and data to be exchanged between them.

### The sheets `SIM6`, `GWF6` `GWT6`

These three spreadsheets in the workbook contain all possible parameters for the modflow 6 `simulation`, its `Groundwater Flow Model` and its `Groundwater Transport Model` with default values. If desired they can be adapted there to the needs of the current case, which is often done for the settings of the solver. But mostly they are left along and overwritten in `mf_adapt.py`, which is imported by `mf_setup.py`.

### The sheet `MP7`

Similarly the sheet `MF7` holds all possible parameters for `Modflpath 7`, which is a program separate from `Modflow 6`.

### The sheet `PER`

This sheet is used to define the stress periods for the current case. It has columns named

`IPER`, `PERLEN`, `NSTP` and `TSMULT`

It will be read in as a pd.DataFrame to make its data available within `mf_adapt.py`.

Note that the stress period number `IPER` is zero based.

One can add as many columns as one likes, for instance one with the data at which each stress period begins.
Of course, it is not necessary to use this sheet, but then the stress periods must be defined by the user in `mf_adapt.py`. Defining them in this spreadsheet has been proven to be convenient.

Missing stress period (gaps in `IPER`) are interpreted as having the same properties as the last one that was defined (= downward filling). The last stress period must always be given as well as the first one. As a convenience, if only the last stress period is specified, then it is assumed that all previous ones have the same properties.

### The sheet `LAY`

This spreadsheet allows specifying layer properties, but of course only those that are constant throughout each layer. Properties that vary throughout the model, mostly layer elevations cannot be specified in a spreadsheet and must be specified by the user in `mf_adapt.py`.

The column heading of the `LAY` sheet will look like this

`LAYER`, `ICELLTYPE`, `k`, `k33`, `Sy`, `Ss`

One may add as many columns as one likes and also leave columns out or ignore them when setting up the model in `mf_adapt.py`. The sheet will be read into a pd.DataFrame and its data are available in `mf_adapt.py`.

The layer numbers are zero-based. Missing layers will be filled in from below. That is, the properties of missing layers are assumed to be the same as the next layer that defines them. Therefore, to specify the same properties for `n` layers all with the same properties one just needs one line defining the properties for layerin `n-1`. That layer number implies how many layers the model wil have (`n`)


## mf_setup.py, the backbone of `mf6lab` for simulations with Modflow in Python

This `mf_setup.py` is used in every case of project and, therefore, is resides in `mf6lab/src`.
mf_setup.py sets up a large dictonary with the package names are keys filling it first with the default parameters from the Excel workbook.

Then it reads local `mf_adapt.py` to adapt values to match the demand for the current simulation.


## mf_adapt.py and mf_settings.py

The files mf_adapt.py and mf_settings.py are local, i.e. they are in `mf6lab/Projects/<project_name>/cases/<case_name>/src`. `mf_run.py` imports `mf_setup.py` to load which packages and modelt to use, together with the parameter. `mf_setup.py` imports `mf_adapt.py` which adapts the variables of the packages to match the actual case. `settings.py` is in turn imported by `mf_adapt.py` to set case-specific properties. `settings.py` mainly minimizes clutter in `mf_adapt.py`, and could be left out. `While` local script `mf_run.py` and `mf6lab`-wide `mf_setup_py` will always be the same for any project and case, the four local files named above are specific to each case. There structure will be more or less the same between most projects, but some adaptations will generally have to be made between cases.

## mf_analyze.py
When starting a new Project, create a folder with the desired project name under `mflab/Projects` Then create a new folder `cases` in that new project folder and then create a new case by creating a new specific case pertaining to that project. One can have an unlimited number of cases under each project. In the new case folder copy the directories of another case and empty them except for the four files mentioined above to allow a fresh start. Rename the workbook to `<case_name>.xlsx`. Then adapt the four files `mf_adapt.py`, `settings.py` and `mf_analyze.py` to represent the new case.

After `Modflow` etc. have (successfully finished), `mf_analyze.py` is run to show the results.

The user should not have to changeneither `mf_run.py` nor `mf_setup.py`.

# Workflow

The basic idea is to prepare your model by adapting the file mf_adapt.py from another case.

mf_adapt.py should als be self running, using a section `if _name__ == "__main__":` which allows showing the grid an other things like printing which packages will be used for verification. This section is never run when mf_adapt is imported by another module like `mf_setup.py`.

`mf_adapt.py` would seldom divert much from previous cases. So look for a similar case and use that as base for your own.

Having `mf_adapt.py` ready and tested, you can run `mf_run.py` which should invoke generation of the input files for 'mf6' and invoke 'mf6' which then runs the simulation.

Assuming Modflow succeeded, it's time to read the (binary) results and analyze and present them.
For this the local file `mf_analyze.py` is used. This file is case specific just like mf_adapt.py. And here too, the current `mf_analyze.py` should seldom deviate much from that of a previous or other similar case. The best approach is to copy an existing file and adapt it.

A project will usually have many cases that are quite similar. Each case-directory contains all data necessary to rerun the simulation in the future. It should also document itself. But the style and extensiveness of the documentation, partly in the docfiles of the modules, partly with comments in the modules and partly in separate reports in the local case or project `doc` directory is, of course, completely up to the user.

@Theo Olshoorn 2O25-11-19
